# 01 - Pipeline de Dataset: Calibración, Descarga y Segmentación de Partidas
Pipeline modular con anclaje universal del temporizador, detección anti-glitch por lookahead y persistencia atómica en Cloudflare R2 vía S3 API.


In [ ]:
# Dependencias esenciales: opencv, easyocr, requests, boto3
# En entornos Kaggle, boto3 y requests ya vienen preinstalados en la imagen base.


In [ ]:
import os, sys, json, time, re
from pathlib import Path
import requests
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import easyocr
import boto3
from botocore.config import Config

# 1. Directorios de trabajo
WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('data')
VIDEOS_DIR = WORK_DIR / 'videos'
SEGMENTS_DIR = WORK_DIR / 'segments'
MANIFESTS_DIR = WORK_DIR / 'manifests'

for d in [VIDEOS_DIR, SEGMENTS_DIR, MANIFESTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 2. Utilidad global de formato de tiempo
def fmt_time(seconds: int) -> str:
    """Convierte segundos a formato MM:SS o H:MM:SS para comparar con YouTube."""
    if not seconds:
        return "0:00"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

# 3. Helper de configuración híbrida (Kaggle Secrets + Variables de Entorno)
def get_config(key: str, default: str = None) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(key)
        if val:
            return val
    except Exception:
        pass
    return os.getenv(key, default)

# 4. Variables globales para el pipeline y comprobaciones visuales
anchored_boxes = {}
processed_results = {}
clips_to_show = []

# 5. Detección de GPU e inicialización de EasyOCR
USE_GPU = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if USE_GPU else 'CPU'
print(f"✔ Directorio de trabajo: {WORK_DIR}")
print(f"✔ Dispositivo de inferencia: {device_name} (GPU activa: {USE_GPU})")

reader = easyocr.Reader(['en'], gpu=USE_GPU)


In [ ]:
# Consulta de videos aprobados, conexión a Cloudflare R2 y filtro de selección

ADMIN_TOKEN = get_config('ADMIN_TOKEN')
ANNOTATOR_BASE = get_config('ANNOTATOR_URL', 'https://cr-dataset.eddux.dev').rstrip('/')
CDN_BASE = get_config('CDN_URL', 'https://cdn-cr.eddux.dev').rstrip('/')

# 1. Configuración de S3 / Cloudflare R2
R2_ENDPOINT = get_config('R2_ENDPOINT')
R2_ACCESS_KEY = get_config('R2_ACCESS_KEY')
R2_SECRET_KEY = get_config('R2_SECRET_KEY')
R2_BUCKET = get_config('R2_BUCKET', 'cr-videos')

s3_client = None
if R2_ENDPOINT and R2_ACCESS_KEY and R2_SECRET_KEY:
    try:
        s3_client = boto3.client(
            's3',
            endpoint_url=R2_ENDPOINT,
            aws_access_key_id=R2_ACCESS_KEY,
            aws_secret_access_key=R2_SECRET_KEY,
            config=Config(signature_version='s3v4')
        )
        print(f"✔ Conectado a Cloudflare R2 vía S3 API (Bucket: '{R2_BUCKET}')")
    except Exception as e:
        print(f"⚠ Error inicializando cliente S3 R2: {e}")
else:
    print("ℹ Modo local: credenciales de R2 no configuradas. Los resultados se guardarán solo en disco local.")

# 2. Consulta de videos aprobados en la base de datos del annotator
headers = {'x-admin-token': ADMIN_TOKEN}
resp = requests.get(f"{ANNOTATOR_BASE}/api/export", headers=headers, timeout=15)
resp.raise_for_status()
all_calibrations = resp.json()
print(f"✔ Total videos aprobados en annotator: {len(all_calibrations)}")

# 3. Detección de videos ya procesados previamente (Idempotencia y tolerancia a fallos)
def get_processed_video_ids():
    """Devuelve el conjunto de video_ids ya segmentados en R2 y en disco local."""
    processed = set()
    for f in SEGMENTS_DIR.glob('*.json'):
        processed.add(f.stem)
    if s3_client:
        try:
            res = s3_client.list_objects_v2(Bucket=R2_BUCKET, Prefix='metadata/segments/')
            for item in res.get('Contents', []):
                vid = Path(item['Key']).stem
                if vid:
                    processed.add(vid)
        except Exception as e:
            print(f"⚠ Error consultando segmentos existentes en R2: {e}")
    return processed

already_processed = get_processed_video_ids()
print(f"✔ Videos ya segmentados previamente (R2 / local): {len(already_processed)}")

# =========================================================================
# CONFIGURACIÓN DE SELECCIÓN DE VIDEOS
# Opciones: 'all', número (ej. 5), o lista ['id1', 'id2']
# =========================================================================
VIDEO_FILTER = 'all'

def select_videos(calibrations_list, filter_rule):
    if filter_rule == 'all':
        return calibrations_list
    if isinstance(filter_rule, int):
        return calibrations_list[:filter_rule]
    if isinstance(filter_rule, str):
        return [c for c in calibrations_list if c['id'] == filter_rule]
    if isinstance(filter_rule, (list, tuple, set)):
        return [c for c in calibrations_list if c['id'] in filter_rule]
    return calibrations_list

selected_videos = select_videos(all_calibrations, VIDEO_FILTER)
pending_count = sum(1 for v in selected_videos if v['id'] not in already_processed)
print(f"✔ Videos seleccionados: {len(selected_videos)} (Pendientes por procesar: {pending_count})")


In [ ]:
# Utilidades de descarga, checkpointing atómico y consolidación de manifiestos

def fetch_video(video_id: str) -> Path:
    """Descarga el video desde Cloudflare CDN si no existe ya en disco."""
    target = VIDEOS_DIR / f"{video_id}.mp4"
    if target.exists() and target.stat().st_size > 1024 * 1024:
        return target

    url = f"{CDN_BASE}/videos/{video_id}.mp4"
    t0 = time.time()
    req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    try:
        with requests.get(url, headers=req_headers, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(target, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        mb = target.stat().st_size / (1024 * 1024)
        print(f"  ✔ {video_id} descargado ({mb:.1f} MB en {time.time()-t0:.1f}s)", flush=True)
        return target
    except Exception as e:
        print(f"  ❌ Error descargando {video_id}: {e}", flush=True)
        return None

def save_segment_checkpoint(video_data: dict):
    """Guarda el JSON del video en disco local y lo sube de forma atómica e inmediata a Cloudflare R2."""
    vid = video_data['video_id']
    # 1. Guardar en disco local de Kaggle
    local_path = SEGMENTS_DIR / f"{vid}.json"
    with open(local_path, 'w', encoding='utf-8') as f:
        json.dump(video_data, f, indent=2, ensure_ascii=False)
        
    # 2. Subir directo a R2 vía S3 API (evitando la caché del CDN)
    if s3_client:
        try:
            s3_key = f"metadata/segments/{vid}.json"
            s3_client.put_object(
                Bucket=R2_BUCKET,
                Key=s3_key,
                Body=json.dumps(video_data, indent=2, ensure_ascii=False).encode('utf-8'),
                ContentType='application/json'
            )
            print(f"  ☁ [{vid}] Segmentación respaldada en R2 ({s3_key})", flush=True)
        except Exception as e:
            print(f"  ⚠ Error subiendo {vid} a R2: {e}", flush=True)

def consolidate_master_manifest():
    """Reúne todas las segmentaciones individuales en el manifiesto global consolidado."""
    all_data = {}
    # Cargar todos los JSONs locales
    for f in SEGMENTS_DIR.glob('*.json'):
        try:
            with open(f, 'r', encoding='utf-8') as jf:
                d = json.load(jf)
                all_data[d['video_id']] = d
        except Exception:
            pass
            
    # Cargar de R2 cualquier otro que no esté en local
    if s3_client:
        try:
            res = s3_client.list_objects_v2(Bucket=R2_BUCKET, Prefix='metadata/segments/')
            for item in res.get('Contents', []):
                vid = Path(item['Key']).stem
                if vid and vid not in all_data:
                    obj = s3_client.get_object(Bucket=R2_BUCKET, Key=item['Key'])
                    all_data[vid] = json.loads(obj['Body'].read().decode('utf-8'))
        except Exception as e:
            print(f"⚠ Error leyendo segmentos de R2 para consolidar: {e}")
            
    total_matches = sum(v.get('total_matches', len(v.get('matches', []))) for v in all_data.values())
    total_sec = sum(sum(m['duration_sec'] for m in v.get('matches', [])) for v in all_data.values())
    
    manifest = {
        'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'total_videos': len(all_data),
        'total_matches': total_matches,
        'total_gameplay_sec': total_sec,
        'total_gameplay_formatted': fmt_time(total_sec),
        'videos': all_data
    }
    
    local_manifest = MANIFESTS_DIR / 'segmented_matches.json'
    with open(local_manifest, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
        
    if s3_client:
        try:
            s3_client.put_object(
                Bucket=R2_BUCKET,
                Key='metadata/segmented_matches.json',
                Body=json.dumps(manifest, indent=2, ensure_ascii=False).encode('utf-8'),
                ContentType='application/json'
            )
            print(f"\n☁ Manifiesto consolidado actualizado en R2: metadata/segmented_matches.json", flush=True)
            print(f"   ({len(all_data)} videos, {total_matches} partidas, {fmt_time(total_sec)} de gameplay)", flush=True)
        except Exception as e:
            print(f"⚠ Error actualizando manifiesto consolidado en R2: {e}")
            
    return manifest


In [ ]:
# =========================================================================
# FASE 1: Localización y Anclaje Inmutable Universal de la Caja del Reloj
# - Búsqueda agnóstica en todo el cuadrante superior derecho amplio (y in [0, 35%], x in [50%, 100%])
# - Estrategia 1: Detección rápida por cluster de dígitos (~5ms)
# - Estrategia 2: OCR completo en el cuadrante extrayendo la bounding box directa de M:SS
# =========================================================================

def read_clock_from_box(box_img: np.ndarray) -> str:
    """Lee los dígitos M:SS de la sub-región inferior de la caja con centrado simétrico."""
    if box_img is None or box_img.size == 0:
        return ""
        
    bh, bw = box_img.shape[:2]
    # Recorte simétrico: 8% a 95% del ancho para no cortar el último dígito
    digits_crop = box_img[int(bh * 0.35):int(bh * 0.96), int(bw * 0.08):int(bw * 0.95)]
    if digits_crop.size == 0:
        return ""
        
    scaled = cv2.resize(digits_crop, (0, 0), fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    res = reader.readtext(scaled, detail=0)
    text = ' '.join(res)
    
    m = re.search(r'([0-5])[\s:\.-bBoO8]?([0-5][0-9])', text)
    if m:
        mins, secs = int(m.group(1)), int(m.group(2))
        return f"{mins}:{secs:02d}"
    return ""

def find_timer_box_universal(frame: np.ndarray):
    """
    Busca la caja del temporizador en el cuadrante superior derecho de forma agnóstica al dispositivo.
    Retorna: (box_coords, clock_str, detection_method)
    """
    h, w = frame.shape[:2]
    qy1, qy2 = 0, int(0.35 * h)
    qx1, qx2 = int(0.50 * w), w
    quad = frame[qy1:qy2, qx1:qx2]
    
    # 1. Estrategia rápida por cluster de dígitos (~5ms)
    gray = cv2.cvtColor(quad, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(quad, cv2.COLOR_BGR2HSV)
    white_mask = (gray > 190).astype(np.uint8) * 255
    red1 = cv2.inRange(hsv, np.array([0, 70, 70]), np.array([12, 255, 255]))
    red2 = cv2.inRange(hsv, np.array([168, 70, 70]), np.array([180, 255, 255]))
    text_mask = cv2.bitwise_or(white_mask, cv2.bitwise_or(red1, red2))
    
    k_w = max(5, int(w * 0.025))
    k_h = max(3, int(h * 0.006))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k_w, k_h))
    closed = cv2.morphologyEx(text_mask, cv2.MORPH_CLOSE, kernel)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    candidates = []
    for c in cnts:
        cx, cy, cw, ch = cv2.boundingRect(c)
        aspect = cw / float(ch) if ch > 0 else 0
        if 1.3 <= aspect <= 4.2 and (0.06 * w) <= cw <= (0.24 * w) and (0.010 * h) <= ch <= (0.050 * h):
            candidates.append((cx, cy, cw, ch))
            
    if candidates:
        candidates.sort(key=lambda b: (b[0] + b[2]), reverse=True)
        for cx, cy, cw, ch in candidates[:2]:
            dx1, dy1 = qx1 + cx, qy1 + cy
            candidate_box = (
                max(0, dx1 - int(0.040 * w)),
                max(0, dy1 - int(0.038 * h)),
                min(w, dx1 + cw + int(0.025 * w)),
                min(h, dy1 + ch + int(0.015 * h))
            )
            clock = read_clock_from_box(frame[candidate_box[1]:candidate_box[3], candidate_box[0]:candidate_box[2]])
            if clock:
                return candidate_box, clock, 'fast_cluster'
                
    # 2. Estrategia OCR en cuadrante amplio con extracción de bounding box
    results = reader.readtext(quad)
    for bbox, text, conf in results:
        m = re.search(r'([0-5])[\s:\.-bBoO8]?([0-5][0-9])', text)
        if m:
            pts = np.array(bbox)
            min_x = int(np.min(pts[:, 0])) + qx1
            max_x = int(np.max(pts[:, 0])) + qx1
            min_y = int(np.min(pts[:, 1])) + qy1
            max_y = int(np.max(pts[:, 1])) + qy1
            bx1 = max(0, min_x - int(0.035 * w))
            bx2 = min(w, max_x + int(0.025 * w))
            by1 = max(0, min_y - int(0.035 * h))
            by2 = min(h, max_y + int(0.015 * h))
            clock_box = (bx1, by1, bx2, by2)
            clock = f"{m.group(1)}:{m.group(2)}"
            return clock_box, clock, 'quadrant_ocr'
            
    return None, "", 'none'

def find_timer_box(frame: np.ndarray):
    """Devuelve la mejor caja estimada para el frame dado."""
    box, _, _ = find_timer_box_universal(frame)
    if box:
        return box
    h, w = frame.shape[:2]
    return (int(w * 0.778), int(h * 0.030), int(w * 0.995), int(h * 0.095))

def calibrate_video_timer(video_path: Path, max_search_sec: int = 60, step_sec: int = 2):
    """Busca y confirma la caja del reloj en los primeros segundos de partida."""
    cap = cv2.VideoCapture(str(video_path))
    calib_result = None
    
    for s in range(2, max_search_sec, step_sec):
        cap.set(cv2.CAP_PROP_POS_MSEC, s * 1000)
        ret, frame = cap.read()
        if not ret:
            continue
        box, clock_text, method = find_timer_box_universal(frame)
        if box and clock_text:
            calib_result = {
                'box': box,
                'sample_sec': s,
                'sample_clock': clock_text,
                'method': method,
                'sample_frame': frame.copy()
            }
            break
                
    cap.release()
    return calib_result


In [ ]:
# =========================================================================
# FASE 2: Algoritmos de Segmentación Continua con Anti-Glitch y Fin por Meseta
# - Detección anti-glitch de nueva partida con confirmación por lookahead
# - Fin exacto por meseta congelada al caer la torre
# - Conexión de tiempo regular con muerte súbita (Overtime)
# =========================================================================

def parse_clock_secs(clock_str: str) -> int:
    """Convierte 'M:SS' a segundos enteros."""
    m, s = map(int, clock_str.split(':'))
    return m * 60 + s

def is_confirmed_new_match(timeline, idx, current_match) -> bool:
    """
    Verifica con lookahead si timeline[idx] es un inicio real de nueva partida:
    1. La partida actual debe llevar al menos 40s jugados.
    2. La lectura actual debe marcar reloj inicial (>= 140s / 2:20).
    3. Confirmación anti-glitch: verifica que la siguiente lectura (idx+1 o idx+2)
       sea consistente con una cuenta regresiva (reloj alto >= 130s y próximo en tiempo).
    """
    if current_match is None:
        return False
    if current_match['last_sec'] - current_match['start_sec'] < 40:
        return False
        
    t_sec, clock = timeline[idx]
    secs = parse_clock_secs(clock)
    if secs < 140:
        return False
        
    # Verificar lookahead para descartar glitches aislados de 1 frame
    if idx + 1 < len(timeline):
        next_t, next_clock = timeline[idx + 1]
        next_secs = parse_clock_secs(next_clock)
        if (next_t - t_sec <= 3) and (130 <= next_secs <= secs + 2):
            return True
        if idx + 2 < len(timeline):
            n2_t, n2_clock = timeline[idx + 2]
            n2_secs = parse_clock_secs(n2_clock)
            if (n2_t - t_sec <= 5) and (120 <= n2_secs <= secs + 2):
                return True
        return False
    return True

def segment_matches_timeline(video_path: Path, box_coords, max_seconds: int = None):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_s = int(total_frames / fps)
    scan_limit = min(duration_s, max_seconds) if max_seconds else duration_s
    x1, y1, x2, y2 = box_coords
    
    timeline = []
    print(f"  Iniciando escaneo a 1 fps (duración: {fmt_time(scan_limit)})...", flush=True)
    for s in range(0, scan_limit):
        cap.set(cv2.CAP_PROP_POS_MSEC, s * 1000)
        ret, frame = cap.read()
        if not ret: break
        
        box_img = frame[y1:y2, x1:x2]
        clock_text = read_clock_from_box(box_img)
        if clock_text:
            timeline.append((s, clock_text))
            
        if s > 0 and s % 300 == 0:
            print(f"    ...escaneados {fmt_time(s)} / {fmt_time(scan_limit)} ({len(timeline)} lecturas de reloj)", flush=True)
            
    cap.release()
    print(f"  Escaneo finalizado: {len(timeline)} lecturas válidas de reloj.", flush=True)
    
    raw_matches = []
    current = None
    
    # 1. Agrupación continua y separación de partidas con lookahead anti-glitch
    for idx, (t_sec, clock) in enumerate(timeline):
        secs = parse_clock_secs(clock)
        
        if current is None:
            if secs >= 140:
                current = {
                    'start_sec': t_sec,
                    'start_clock': clock,
                    'last_sec': t_sec,
                    'in_overtime': False,
                    'readings': [(t_sec, clock)]
                }
        else:
            dt = t_sec - current['last_sec']
            prev_secs = parse_clock_secs(current['readings'][-1][1])
            
            # A. Comprobar si se confirma el inicio de una NUEVA partida
            if is_confirmed_new_match(timeline, idx, current):
                if current['last_sec'] - current['start_sec'] >= 40:
                    raw_matches.append(current)
                current = {
                    'start_sec': t_sec,
                    'start_clock': clock,
                    'last_sec': t_sec,
                    'in_overtime': False,
                    'readings': [(t_sec, clock)]
                }
                continue
                
            # B. Descartar lecturas aberrantes aisladas (glitches que saltan >45s sin ser overtime)
            if secs > prev_secs + 45 and not (prev_secs <= 10 and secs <= 125):
                continue
                
            # C. Detectar si entró en Overtime (salto de <=10s a 55s-125s)
            if prev_secs <= 10 and (55 <= secs <= 125):
                current['in_overtime'] = True
                
            # D. Continuidad temporal
            if dt <= 5:
                current['last_sec'] = t_sec
                current['readings'].append((t_sec, clock))
            else:
                if current['last_sec'] - current['start_sec'] >= 40:
                    raw_matches.append(current)
                if secs >= 140:
                    current = {
                        'start_sec': t_sec,
                        'start_clock': clock,
                        'last_sec': t_sec,
                        'in_overtime': False,
                        'readings': [(t_sec, clock)]
                    }
                else:
                    current = None
                    
    if current and (current['last_sec'] - current['start_sec'] >= 40):
        raw_matches.append(current)
        
    # 2. Refinamiento del final exacto por meseta
    final_matches = []
    for m_item in raw_matches:
        reads = m_item['readings']
        exact_end_sec = m_item['last_sec']
        exact_end_clock = reads[-1][1]
        
        tail_start = max(0, len(reads) - 20)
        for i in range(tail_start, len(reads) - 2):
            if reads[i][1] == reads[i+1][1] == reads[i+2][1]:
                exact_end_sec = reads[i][0]
                exact_end_clock = reads[i][1]
                break
                
        m_item['end_sec'] = exact_end_sec
        m_item['end_clock'] = exact_end_clock
        final_matches.append(m_item)
        
    return timeline, final_matches


In [ ]:
# =========================================================================
# BUCLE DE EJECUCIÓN PIPELINE: VIDEO POR VIDEO CON TOLERANCIA A FALLOS
# - Si el video ya está procesado en R2 o local, lo salta en 0 segundos.
# - Procesa Fase 1 y Fase 2, guarda checkpoint atómico y sube a R2.
# - Borra el .mp4 temporal para liberar disco en Kaggle (manteniendo previews).
# =========================================================================

SKIP_ALREADY_PROCESSED = True
DELETE_VIDEO_AFTER_PROCESSING = True

print(f"Iniciando pipeline para {len(selected_videos)} videos seleccionados...\n", flush=True)

for v_idx, v_info in enumerate(selected_videos, 1):
    vid = v_info['id']
    v_title = v_info.get('title', '')[:45]
    v_dur = fmt_time(v_info.get('duration_s', 0))
    
    # 1. Comprobar si ya está procesado previamente
    if SKIP_ALREADY_PROCESSED and vid in already_processed:
        local_f = SEGMENTS_DIR / f"{vid}.json"
        if local_f.exists():
            try:
                with open(local_f, 'r', encoding='utf-8') as jf:
                    cached_data = json.load(jf)
                    processed_results[vid] = cached_data
                    anchored_boxes[vid] = {
                        'box': cached_data['timer_box'],
                        'sample_sec': cached_data.get('sample_sec', 0),
                        'sample_clock': cached_data.get('sample_clock', ''),
                        'method': cached_data.get('detection_method', 'cached')
                    }
            except Exception:
                pass
        print(f"[{v_idx}/{len(selected_videos)}] ⏭ [{vid}] Ya procesado previamente, saltando...", flush=True)
        continue
        
    print(f"\n=======================================================", flush=True)
    print(f"[{v_idx}/{len(selected_videos)}] Procesando: [{vid}] {v_title} ({v_dur})", flush=True)
    print(f"=======================================================", flush=True)
    
    # 2. Descargar video individual
    vpath = fetch_video(vid)
    if not vpath or not vpath.exists():
        print(f"  ❌ No se pudo descargar el video {vid}. Saltando...", flush=True)
        continue
        
    t_start_v = time.time()
    
    # 3. Fase 1: Calibrar y anclar caja del temporizador
    calib = calibrate_video_timer(vpath)
    if not calib:
        print(f"  ⚠ [{vid}] No se pudo anclar la caja del reloj en los primeros 60s. Saltando...", flush=True)
        if DELETE_VIDEO_AFTER_PROCESSING:
            vpath.unlink(missing_ok=True)
        continue
        
    anchored_boxes[vid] = calib
    print(f"  ✔ [{vid}] Caja anclada en {calib['box']} (Reloj: {calib['sample_clock']} vía {calib['method']})", flush=True)
    
    # 4. Fase 2: Segmentar partidas a 1 fps
    tl, m_list = segment_matches_timeline(vpath, calib['box'])
    
    # 5. Capturar miniaturas de inicio y fin para la galería antes de borrar el video
    cap = cv2.VideoCapture(str(vpath))
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    for p_idx, m in enumerate(m_list, 1):
        s_sec, e_sec = m['start_sec'], m['end_sec']
        cap.set(cv2.CAP_PROP_POS_MSEC, s_sec * 1000)
        _, f_start = cap.read()
        cap.set(cv2.CAP_PROP_POS_MSEC, e_sec * 1000)
        _, f_end = cap.read()
        ov_label = " + Overtime" if m.get('in_overtime') else ""
        clips_to_show.append({
            'title': f"{vid} (P{p_idx}{ov_label})",
            'start_info': f"Inicio {fmt_time(s_sec)} (reloj {m['start_clock']})",
            'end_info': f"Fin {fmt_time(e_sec)} (reloj {m['end_clock']})",
            'start_img': f_start,
            'end_img': f_end
        })
    cap.release()
    
    # 6. Empaquetar y guardar checkpoint atómico (Local + Cloudflare R2 vía S3 API)
    video_result = {
        'video_id': vid,
        'title': v_info.get('title', ''),
        'duration_s': v_info.get('duration_s', 0),
        'resolution': [frame_w, frame_h],
        'timer_box': calib['box'],
        'detection_method': calib['method'],
        'sample_sec': calib['sample_sec'],
        'sample_clock': calib['sample_clock'],
        'total_matches': len(m_list),
        'processed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'matches': [
            {
                'match_id': f"{vid}_p{idx:02d}",
                'match_index': idx,
                'start_sec': m['start_sec'],
                'end_sec': m['end_sec'],
                'duration_sec': m['end_sec'] - m['start_sec'],
                'start_clock': m['start_clock'],
                'end_clock': m['end_clock'],
                'in_overtime': bool(m.get('in_overtime', False)),
                'youtube_url': f"https://youtu.be/{vid}?t={m['start_sec']}"
            }
            for idx, m in enumerate(m_list, 1)
        ]
    }
    
    save_segment_checkpoint(video_result)
    processed_results[vid] = video_result
    already_processed.add(vid)
    
    # 7. Liberar espacio en disco eliminando el video .mp4 procesado
    if DELETE_VIDEO_AFTER_PROCESSING:
        vpath.unlink(missing_ok=True)
        print(f"  🗑 Archivo temporal {vid}.mp4 eliminado (disco liberado)", flush=True)
        
    print(f"✔ [{vid}] Completado en {time.time()-t_start_v:.1f}s ({len(m_list)} partidas detectadas):", flush=True)
    for idx, m in enumerate(video_result['matches'], 1):
        ov_str = " (incluye Overtime)" if m['in_overtime'] else ""
        print(f"    P{idx}: {fmt_time(m['start_sec'])} ({m['start_clock']}) -> {fmt_time(m['end_sec'])} ({m['end_clock']}) | Duración: {fmt_time(m['duration_sec'])}{ov_str}", flush=True)
        print(f"        Link YouTube: {m['youtube_url']}", flush=True)

# 8. Consolidación final del manifiesto maestro con todos los videos
master_manifest = consolidate_master_manifest()


In [ ]:
# Comprobación Visual de Cajas Ancladas (Fase 1)
if anchored_boxes:
    n = len(anchored_boxes)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)
        
    for idx, (vid, data) in enumerate(anchored_boxes.items()):
        if 'sample_frame' not in data:
            continue
        f = data['sample_frame'].copy()
        x1, y1, x2, y2 = data['box']
        
        cv2.rectangle(f, (x1, y1), (x2, y2), (0, 255, 0), 4)
        
        axes[idx, 0].imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        axes[idx, 0].set_title(f"[{vid}] Frame a t={data['sample_sec']}s")
        axes[idx, 0].axis('off')
        
        crop_box = data['sample_frame'][y1:y2, x1:x2]
        axes[idx, 1].imshow(cv2.cvtColor(crop_box, cv2.COLOR_BGR2RGB))
        method_str = data.get('method', 'auto')
        axes[idx, 1].set_title(f"Lectura: '{data['sample_clock']}' ({method_str})")
        axes[idx, 1].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No hay nuevas cajas ancladas para visualizar en esta sesión.")


In [ ]:
# Comprobación Visual de Clips Segmentados (Inicio vs Fin)
# Utiliza las miniaturas en memoria capturadas durante la segmentación
if clips_to_show:
    max_previews = min(12, len(clips_to_show))
    print(f"Mostrando {max_previews} de {len(clips_to_show)} clips segmentados en esta sesión:")
    fig, axes = plt.subplots(max_previews, 2, figsize=(10, 4.5 * max_previews))
    if max_previews == 1:
        axes = np.expand_dims(axes, 0)
        
    for idx in range(max_previews):
        c = clips_to_show[idx]
        axes[idx, 0].imshow(cv2.cvtColor(c['start_img'], cv2.COLOR_BGR2RGB))
        axes[idx, 0].set_title(f"{c['title']}\n{c['start_info']}")
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(cv2.cvtColor(c['end_img'], cv2.COLOR_BGR2RGB))
        axes[idx, 1].set_title(f"{c['title']}\n{c['end_info']}")
        axes[idx, 1].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No hay nuevos clips para mostrar en esta sesión (los videos seleccionados ya estaban procesados).")
